# Tours on a layered graph, with CSR matrices: a walkthrough

This notebook follows the whole process on the toy OD (6 zones, 3 time periods) with the code in `layered/layered_sparse.py`, one step at a time. The part that is hardest to see is the **CSR** (compressed sparse row) storage, so it comes first and slowly (parts 2 and 3). Everything else is the algorithm you already know from the tutorial: forward and backward BFS, trim, count, the Markov chain and sampling.

| part | what |
|---|---|
| 1 | The OD as a list of edges |
| 2 | **CSR from scratch**: three arrays instead of an N x N table |
| 3 | **A matrix-vector product is one pass over the edges** (= one BFS level) |
| 4 | Time periods: waiting is a cumulative sum |
| 5 | The legs of one chain |
| 6 | Forward, backward, trim |
| 7 | Counting tours |
| 8 | The Markov chain |
| 9 | Sampling |
| 10 | Check against the dense numpy version |
| 11 | What changes on Berlin |

**Run it with uv** (from the folder holding this file):

```
uv run --no-project --with jupyterlab --with pandas --with numpy --with scipy jupyter lab layered_walkthrough.ipynb
```

On a remote machine add `--no-browser --port 8888` and forward the port with `ssh -L 8888:localhost:8888 <host>`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse

sys.path.insert(0, str(Path("layered")))
import layered_sparse as ls

pd.set_option("display.width", 120)

## 1. The OD as a list of edges

One row per OD edge: origin, destination, purpose, direction, time period and trips. The toy has three periods, `0` (morning), `1` (midday) and `2` (evening). Zones `A` and `B` are residential; `W` is work, `M` a mall, `P` a park and `G` a gym.

Everything below is built from this table.

In [ ]:
raw = pd.read_csv("layered/reference/toy_od_tp.csv", dtype={"tp": str})
od = raw.rename(columns={"orig": "Orig", "dest": "Dest", "purpose": "Purpose",
                         "direction": "Direction", "tp": "Timeperiod", "trips": "Trips"})
TP_NAMES = ["AM", "MD", "PM"]          # periods 0, 1, 2
print(len(od), "edges,", od.Trips.sum(), "trips")
od.head(8)

`Layers` turns that table into sparse matrices: for every `(purpose, direction)` and every period, one matrix with a row per origin zone and a column per destination zone. We number the zones alphabetically (`A`=0, `B`=1, `G`=2, `M`=3, `P`=4, `W`=5).

In [ ]:
layers = ls.Layers(od, ["0", "1", "2"])
ZONES, N, K = layers.zones, layers.N, layers.K
print("zones:", dict(enumerate(ZONES)), "| N =", N, "| K =", K)
print("matrices:", sorted(layers.S))
m = layers.S["NHBW", "NonHome"][1]      # non-home work trips leaving in period 1 (midday)
print(type(m).__name__, m.shape, "with", m.nnz, "stored values out of", N * N)

## 2. CSR from scratch

Take the non-home work trips of the midday period. As a table (rows = origin, columns = destination) it is mostly zeros:

In [ ]:
pd.DataFrame(m.toarray(), index=ZONES, columns=ZONES).astype(int)

For Berlin the same table has 1,291 x 1,291 = 1.7 million cells and, in the busiest period, about 1,300 non-zero ones. CSR keeps **only the edges**, sorted by origin, in three flat arrays:

* `data`: the trips of each edge, in order;
* `indices`: the destination zone of each edge;
* `indptr` ("index pointer"): where each origin's edges start. The edges leaving zone `z` are those at positions `indptr[z]` up to (not including) `indptr[z + 1]`.

Let's build them by hand from the edge list.

In [ ]:
edges = od[(od.Purpose == "NHBW") & (od.Direction == "NonHome") & (od.Timeperiod == "1")]
zone_id = {z: i for i, z in enumerate(ZONES)}
o = edges.Orig.map(zone_id).to_numpy()
d = edges.Dest.map(zone_id).to_numpy()
order = np.lexsort((d, o))                       # sort by origin, then destination

data = edges.Trips.to_numpy(float)[order]        # 1. trips of the edges, sorted by origin
indices = d[order]                               # 2. destination of each edge
per_origin = np.bincount(o, minlength=N)         # how many edges leave each zone
indptr = np.concatenate([[0], np.cumsum(per_origin)])   # 3. running total = start of each row

print("data   ", data)
print("indices", indices, "->", [ZONES[i] for i in indices])
print("per_origin", per_origin, "(edges leaving", ZONES, ")")
print("indptr ", indptr)

# scipy builds exactly the same arrays
assert (m.data == data).all() and (m.indices == indices).all() and (m.indptr == indptr).all()

`indptr` has N + 1 entries and is just the cumulative count of edges per origin, starting at 0. Now the key trick: **the edges leaving zone `z` are a contiguous slice**, so finding them costs nothing (no search, no hash lookup).

In [ ]:
rows = []
for z in range(N):
    lo, hi = indptr[z], indptr[z + 1]          # the slice of zone z
    rows.append({"zone": ZONES[z], "slice": f"{lo}:{hi}",
                 "destinations": [ZONES[i] for i in indices[lo:hi]], "trips": data[lo:hi].astype(int).tolist()})
pd.DataFrame(rows).set_index("zone")

A zone with no outgoing edges simply has an empty slice (`lo == hi`): that is why zones `A` and `B` show `[]` above. `ls.row(m, z)` does exactly this slicing and returns `(destinations, trips)`.

Compare the cost: a dense table is N x N numbers however few trips there are. CSR is `2 x (number of edges) + N + 1` numbers.

In [ ]:
print("dense :", N * N, "numbers")
print("CSR   :", len(data) + len(indices) + len(indptr), "numbers")
print(ls.row(m, zone_id["W"]))

## 3. A matrix-vector product is one pass over the edges

A **vector** over the zones says something about each zone, for example "how many travellers are there". Moving them along the edges of the leg gives a new vector. With the CSR arrays that is a loop over the edges, written out:

```python
for z in range(N):
    for e in range(indptr[z], indptr[z + 1]):        # edges leaving z
        out[indices[e]] += data[e] * v[z]            # push v along the edge
```

This is `m.T @ v`. It is what one level of a forward BFS does: it takes everyone in the current set of nodes and moves them across all edges.

In [ ]:
def push(indptr, indices, data, v):
    """Forward: from the origin of every edge to its destination (= m.T @ v)."""
    out = np.zeros(len(v))
    for z in range(len(v)):
        for e in range(indptr[z], indptr[z + 1]):
            out[indices[e]] += data[e] * v[z]
    return out


v = np.zeros(N)
v[zone_id["W"]] = 1                              # one traveller, at work
print("push :", dict(zip(ZONES, push(indptr, indices, data, v).tolist())))
print("m.T@v:", dict(zip(ZONES, (m.T @ v).tolist())))
assert np.allclose(push(indptr, indices, data, v), m.T @ v)

Read the output as "after one midday leg from W, the trips lead to W (itself), M, P and G, weighted by trips". With 0/1 values instead of trips, `> 0` is simply **which zones can be reached**: a BFS level.

The **backward** pass asks the opposite question: *which zones have an edge into the set `v`?* Now each zone *collects* from the destinations of its own edges:

```python
for z in range(N):
    for e in range(indptr[z], indptr[z + 1]):
        out[z] += data[e] * v[indices[e]]            # pull v back along the edge
```

This is `m @ v`, a BFS on the reversed graph. We never build a reversed graph: reading the same arrays the other way round is enough.

In [ ]:
def pull(indptr, indices, data, v):
    """Backward: from the destination of every edge to its origin (= m @ v)."""
    out = np.zeros(len(v))
    for z in range(len(v)):
        for e in range(indptr[z], indptr[z + 1]):
            out[z] += data[e] * v[indices[e]]
    return out


target = np.zeros(N)
target[zone_id["G"]] = 1                         # "who can reach the gym in one leg?"
print("pull :", dict(zip(ZONES, pull(indptr, indices, data, target).tolist())))
print("m@v  :", dict(zip(ZONES, (m @ target).tolist())))
assert np.allclose(pull(indptr, indices, data, target), m @ target)

So both sweeps are the same loop over the edges, only the direction of the assignment differs:

| | loop body | scipy | meaning |
|---|---|---|---|
| forward (push) | `out[dest] += w * v[orig]` | `m.T @ v` | where can we get to / how many ways |
| backward (pull) | `out[orig] += w * v[dest]` | `m @ v` | who can get there / probability of getting there |

Both cost one pass over the edges, `O(edges)`, not `O(N^2)`. The values decide what is computed:

* booleans (`> 0`): reachability, the BFS of the tutorial;
* path counts: the number of tours;
* trips times a probability: the Markov chain of part 8.

Also note that CSR is the natural format for `pull` (and for sampling in part 9): all edges of one origin are next to each other. For `push` scipy transposes (`m.T` is a CSC view of the same arrays, no copy).

## 4. Time periods: waiting is a cumulative sum

A node is a zone and **the period of the last leg**. A vector over one layer is an array `[zone, period]`. Each leg has one matrix **per period** (the edge list of that period), and `ls.by_period(mats, x)` multiplies matrix `t` with column `t` of `x`.

The only new rule is waiting: after arriving in period `t` you may leave in `t` or in any later period. For "can I leave in period `t`" that is a cumulative OR (or sum) along the period axis, `ls.since`. Going backwards, "will I still be able to leave in a period >= t" is the same cumulative operation in reverse, `ls.from_now`.

In [ ]:
arrived = np.zeros((2, K), dtype=bool)
arrived[0, 0] = True                              # zone 0 reached in period 0
arrived[1, 1] = True                              # zone 1 reached in period 1
print("arrived (rows = zones, columns = periods)\n", arrived.astype(int))
print("since    (may leave in period t)\n", ls.since(arrived).astype(int))

can_leave = np.array([[0, 0, 1], [0, 1, 0]], dtype=bool)   # edges available from each zone, per period
print("can_leave\n", can_leave.astype(int))
print("from_now (arrived in period t: may still leave later)\n", ls.from_now(can_leave).astype(int))

## 5. The legs of one chain

We pick home zone `A` and the chain Home > Work > Other > Work > Home. Its four legs have the purposes HBW, NHBW, NHBW, HBW; the first one is a from-home edge, the last a to-home edge, the others non-home.

`layers.legs` returns `W[leg][period]`, a CSR matrix per leg and period holding the trips. Leg 0 keeps only the rows of the home zone. Below, the number of stored edges per leg and period:

In [ ]:
HOME = zone_id["A"]
ACTS = ["Home", "Work", "Other", "Work", "Home"]
print("purposes:", [ls.purpose_of(a, b) for a, b in zip(ACTS, ACTS[1:])])

W = layers.legs(ACTS, HOME)
pd.DataFrame([[mats[t].nnz for t in range(K)] for mats in W],
             index=[f"leg {i + 1}" for i in range(len(W))], columns=TP_NAMES)

`connectivity` turns the trips into 0/1 and restricts the last leg to arrive at the home zone. The weights `W` are kept for the probabilities: the edges into the *other* home zone are dead ends, but a traveller choosing among edges still has them available.

In [ ]:
C = ls.connectivity(W, HOME)
print("edges per leg in C:", [sum(m.nnz for m in mats) for mats in C], "| in W:", [sum(m.nnz for m in mats) for mats in W])

def node(z, t): return f"{ZONES[z]}@{TP_NAMES[t]}"
def edge_list(mats):
    """The edges of one leg as text, read from the CSR arrays of every period."""
    return [f"{ZONES[z]}>{ZONES[d]}@{TP_NAMES[t]}"
            for t, m in enumerate(mats) for z in range(N) for d in ls.row(m, z)[0]]
for i, mats in enumerate(C, start=1):
    print(f"leg {i}:", " ".join(edge_list(mats)))

## 6. Forward, backward, trim

* `forward`: `fwd[i][z, t]` is True if node (z, t) can be reached from home in `i` legs. Each layer is: wait (`since`), then push along the edges (`by_period(..., transpose=True)`).
* `backward`: `bwd[i][z, t]` is True if from node (z, t) of layer `i` home can still be reached. Each layer is: pull from the next layer (`by_period`), then allow waiting (`from_now`).
* `trim` keeps the edges whose origin can be reached by the edge's period and whose destination can still get home.

In [ ]:
fwd = ls.forward(C, HOME)
bwd = ls.backward(C, HOME)
Ct = ls.trim(C, fwd, bwd)

def nodes(a): return " ".join(node(z, t) for z, t in zip(*np.nonzero(a)))
for i in range(len(fwd)):
    print(f"layer {i}   reached from home: {nodes(fwd[i]) or '-'}")
    print(f"          can get home    : {nodes(bwd[i])}")
print()
print("edges before trimming:", sum(m.nnz for mats in C for m in mats), "| after:", sum(m.nnz for mats in Ct for m in mats))
for i, mats in enumerate(Ct, start=1):
    print(f"leg {i}:", " ".join(edge_list(mats)))

## 7. Counting tours

A tour is a path from `(A, period 0)` in layer 0 to the home zone in layer 4. Instead of listing paths we count them: `paths[z, t]` is the number of ways to be at node (z, t); one leg later it is the sum over incoming edges, which is exactly the forward product of part 3 with the path counts as the vector (after `cumsum` for waiting).

In [ ]:
n_tours = ls.count_tours(Ct, HOME)
tours = ls.list_tours(Ct, HOME)       # fine here; on real data we count instead
print("counted:", n_tours, "| listed:", len(tours))
assert n_tours == len(tours)

def label(tour): return " ".join(node(z, t) if i else ZONES[z] for i, (z, t) in enumerate(tour))
print(label(tours[0])); print(label(tours[-1]))

## 8. The Markov chain

From node `(z, t)` a traveller picks one of the edges leaving `z` in a period `>= t`, in proportion to its trips. `beta[i][z, t]` is the probability of getting home in the remaining legs; `den[i][z, t]` is the total of the trips among which the choice is made. Both sweeps run backwards with the *pull* of part 3, now with trips and probabilities as values instead of booleans:

```
beta[i][z, t] = sum over edges (z -> d, period u >= t) of  trips * beta[i+1][d, u]   /   sum of trips
```

Edges into dead-end zones stay in the denominator (they are part of `W`) and contribute 0 to the numerator.

In [ ]:
beta, den = ls.completion(W, HOME)
beta0 = beta[0][HOME, 0]
print("probability that a walk from home gets home:", round(beta0, 4))
print("beta at the last layer:\n", pd.DataFrame(beta[-1], index=ZONES, columns=TP_NAMES).astype(int))
pd.DataFrame(beta[-2], index=ZONES, columns=TP_NAMES).round(3)

The probability of a complete tour is the product of its edge probabilities, divided by `beta0` because we only look at walks that get home:

```
P(tour) = prod( trips(edge) / den[i][z, t] ) / beta0
```

In [ ]:
probs = pd.Series({label(t): ls.tour_probability(W, den, t, beta0) for t in tours}).sort_values(ascending=False)
print("sum of probabilities:", round(probs.sum(), 10))
probs.head(8).round(4)

## 9. Sampling

To draw a tour, walk from home and at every step pick an edge with probability proportional to `trips * beta(next node)`. Edges that lead to a dead end have `beta = 0` and are never chosen, so every walk gets home, and the draws follow exactly the probabilities above.

This is where CSR pays off a second time: the candidate edges of the current zone are one **slice** of each later period's arrays. Here are the options at the first step by hand:

In [ ]:
z, t = HOME, 0                                       # start: home zone, period 0
options = []
for u in range(t, K):                                # every period we may leave in
    dests, trips = ls.row(W[0][u], z)                # CSR slice: edges of z in period u
    for dest, w in zip(dests, trips):
        options.append({"edge": f"{ZONES[z]}>{ZONES[dest]}@{TP_NAMES[u]}", "trips": w,
                        "beta(next)": beta[1][dest, u]})
step = pd.DataFrame(options)
step["weight"] = step.trips * step["beta(next)"]
step["pick prob."] = step.weight / step.weight.sum()
step.round(3)

In [ ]:
rng = np.random.default_rng(0)
draws = pd.Series([label(ls.sample_tour(W, beta, HOME, rng)) for _ in range(20000)]).value_counts(normalize=True)
pd.DataFrame({"exact": probs, "sampled": draws}).head(8).round(4)

## 10. Check against the dense numpy version

`layered/reference/tours_numpy_tp.py` does the same with full `N x N x K` arrays and no CSR. The two must agree on everything: tour count, the tours, `beta0` and every tour probability.

In [ ]:
sys.path.insert(0, str(Path("layered/reference")))
import tours_numpy_tp as dense

zones_d, tps_d, T = dense.read_od("layered/reference/toy_od_tp.csv")
assert zones_d == ZONES
Wd = dense.leg_weights(T, ACTS, HOME)
Ctd = dense.trim(dense.connectivity(Wd, HOME), HOME)
beta_d, den_d = dense.completion(Wd, HOME)
tours_d = dense.list_tours(Ctd, HOME)

assert dense.count_tours(Ctd, HOME) == n_tours
assert {tuple(t) for t in tours} == {tuple((int(z), int(u)) for z, u in t) for t in tours_d}
assert abs(beta_d[0][HOME, 0] - beta0) < 1e-12
assert all(abs(dense.tour_probability(Wd, den_d, t, beta_d[0][HOME, 0])
               - ls.tour_probability(W, den, t, beta0)) < 1e-12 for t in tours_d)
print("sparse and dense agree on", n_tours, "tours")

## 11. What changes on Berlin

The code above is what runs on Berlin; only the sizes differ (H3 resolution 8, 1,291 zones, 7 periods, 58,924 OD edges):

| | dense (`N x N x K`) | sparse (CSR per period) |
|---|---|---|
| memory for the 8 purpose and direction matrices | 0.75 GB | 1 MB |
| one home zone and chain (4 to 5 legs) | 1 to 2.4 s | 20 to 33 ms |
| all 1,044 home zones, one chain | 20 to 40 min (estimated) | about 30 s |

`layered/chains_berlin.py` counts the tours of every chain of the truth and checks that each truth tour is a path of the graph (26,480 of 26,480 are). Run it with `uv run --group validation python layered/chains_berlin.py` from the repository root; it needs a validation run directory.

The counts of the whole city, with no distance cap and no distribution filter:

| chain | legs | tours |
|---|---|---|
| Home > Work > Other > Work > Home | 4 | 20,753 |
| Home > Work > Other > Other > Work > Home | 5 | 88,095 |
| Home > Other > Other > Other > Home | 4 | 58,996 |
| Home > Other > Other > Other > Other > Home | 5 | 298,089 |
| Home > Other > Other > Other > Other > Other > Home | 6 | 1,550,812 |